# De un producto a la búsqueda

Este notebook recorre el mismo camino que usa Lente:

1. arma el texto de un producto y descarga su foto
2. pide tres embeddings a Gemini: fusionado, solo texto y solo imagen
3. guarda el punto en Qdrant sin recrear la colección
4. embebe la consulta gafas de sol y busca por similitud coseno

Cohere sigue la misma forma (embed_document, embed_document_text, embed_document_image, embed_query) sobre el vector nombrado cohere. Aquí usamos Gemini para ver el proceso una sola vez.

Corre las celdas desde `backend/` o desde `backend/notebooks/`. Hace falta Qdrant en `localhost:6333`, las claves en `backend/.env`, y el catálogo ya cargado con el notebook de seed.

Las celdas de código vienen colapsadas. Haz clic en la barra izquierda de una celda (o en el chevron) para abrirla cuando quieras ver el detalle.



In [ ]:
import sys
import uuid
from pathlib import Path

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT / "backend", ROOT.parent / "backend"):
    if (candidate / "app").exists():
        BACKEND = candidate
        break
else:
    raise RuntimeError("Run this notebook from backend/ or from backend/notebooks/.")

sys.path.insert(0, str(BACKEND))

import httpx
from qdrant_client import AsyncQdrantClient
from qdrant_client.http import models

from app.config import Settings
from app.domains.catalog.domain.product import Product, document_text
from app.domains.catalog.infrastructure.gemini_embedder import GeminiEmbedder
from app.domains.catalog.infrastructure.qdrant_catalog import QdrantCatalog
from app.ssl import use_system_certificates

use_system_certificates()

settings = Settings()
http = httpx.AsyncClient(timeout=60.0, follow_redirects=True)
qdrant = AsyncQdrantClient(url=settings.qdrant_url)
gemini = GeminiEmbedder(settings, http)
repository = QdrantCatalog(qdrant, settings.collection_name, settings.embedding_dimension)

NAMESPACE = uuid.UUID("8d8b5b1e-6c3a-4f0e-9a1d-3c2b7e4f5a60")
print("backend", BACKEND)
print("collection", settings.collection_name)
print("dimension", settings.embedding_dimension)


## 1. Un producto del catálogo

Tomamos LN3548, unas gafas de sol. El texto que se embebe no es solo el nombre: document_text junta nombre, etiqueta de categoría y descripción. Ese string es el que entra al modelo junto con la foto.


In [ ]:
product = Product(
    id="sol-aviador",
    name="Gafas De Sol Lente LN3548 Verde/Dorado",
    category="gafas",
    price=156,
    description="Metal dorado. Forma redonda. Unisex. Lunas verdes, no polarizadas. Graduables.",
    image_url="https://images.unsplash.com/photo-1511499767150-a48a237f0083?auto=format&fit=crop&w=1200&q=80",
)

text = document_text(product)
response = await http.get(product.image_url)
response.raise_for_status()
image = response.content
mime_type = response.headers.get("content-type", "image/jpeg").split(";")[0].strip()

print(text)
print("image bytes", len(image), mime_type)


## 2. Encoding

embed_document pide un vector search_document con texto y foto a la vez. Ese es el que usa el ranking.

Los otros dos son solo texto y solo imagen. Van en el payload para el panel de match, no como vectores de búsqueda.


In [ ]:
fused = await gemini.embed_document(text, image, mime_type)
text_only = await gemini.embed_document_text(text)
image_only = await gemini.embed_document_image(image, mime_type)

def preview(name: str, vector: list[float]) -> None:
    """Print dimension and the first four floats."""
    head = ", ".join(f"{value:.4f}" for value in vector[:4])
    print(f"{name}: dim={len(vector)} head=[{head}, ...]")

preview("fused (text + image)", fused)
preview("text only", text_only)
preview("image only", image_only)


## 3. Carga en Qdrant

El punto lleva dos vectores nombrados: gemini (fusionado) y cohere. Si el producto ya existe, leemos el cohere actual para no borrarlo al hacer upsert.

En el payload van el producto y los vectores sueltos de texto e imagen. Este paso no recrea la colección.


In [ ]:
point_id = str(uuid.uuid5(NAMESPACE, product.id))

existing = await qdrant.retrieve(
    collection_name=settings.collection_name,
    ids=[point_id],
    with_payload=True,
    with_vectors=True,
)

cohere_vector: list[float] | None = None
cohere_text = None
cohere_image = None
if existing:
    point = existing[0]
    vectors = point.vector or {}
    if isinstance(vectors, dict) and "cohere" in vectors:
        values = vectors["cohere"]
        if hasattr(values, "tolist"):
            values = values.tolist()
        cohere_vector = [float(item) for item in values]
    payload = point.payload or {}
    cohere_text = payload.get("cohere_text")
    cohere_image = payload.get("cohere_image")

if cohere_vector is None:
    raise RuntimeError(
        "This product has no Cohere vector yet. Run backend/scripts/seed_products.ipynb first."
    )

payload = {
    **product.to_dict(),
    "gemini_text": text_only,
    "gemini_image": image_only,
    "cohere_text": cohere_text,
    "cohere_image": cohere_image,
}

await qdrant.upsert(
    collection_name=settings.collection_name,
    points=[
        models.PointStruct(
            id=point_id,
            vector={"gemini": fused, "cohere": cohere_vector},
            payload=payload,
        )
    ],
    wait=True,
)

print("upserted", product.id, point_id)
print("named vectors: gemini, cohere")
print("payload keys:", ", ".join(sorted(payload)))


## 4. Consulta y similitud

La búsqueda embebe gafas de sol con search_query y compara ese vector con el gemini fusionado de cada producto. El score es similitud coseno: 1 es idéntico, 0 es ortogonal.


In [ ]:
query = "gafas de sol"
query_vector = await gemini.embed_query(query, None, None)
preview("query", query_vector)

hits = await repository.search(query_vector, "gemini", limit=5)
print()
print(f"query: {query!r}")
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. {hit.score:.4f}  {hit.product.name}")


## Cierre

- Documento: texto + foto → vector fusionado (search_document), más texto e imagen sueltos.
- Qdrant: ranking sobre vectores nombrados; modalidades en el payload.
- Consulta: search_query → búsqueda coseno sobre el vector del proveedor.

Cuando termines, cierra los clientes:


In [ ]:
await http.aclose()
await qdrant.close()
print("closed")
